# Notebook 11: Unsloth Setup & Model Loading

## Overview

- **Duration**: ~1.5 hours
- **Prerequisites**: Notebook 10 (LoRA Theory)
- **Learning Objectives**:
  1. Understand why Unsloth is faster than standard fine-tuning
  2. Install and configure Unsloth
  3. Load a 4-bit quantized model
  4. Add LoRA adapters using Unsloth

## Introduction

### Why Unsloth?

Unsloth provides:
- **2x faster** training than HuggingFace + PEFT
- **70% less memory** usage
- **No accuracy loss** compared to full precision training
- Optimized kernels for attention, MLP, and embedding layers

### Supported Models

| Model Family | Sizes | Notes |
|-------------|-------|-------|
| Llama 3.2 | 1B, 3B | Recommended, excellent quality |
| Llama 3.1 | 8B, 70B | More capable, higher VRAM |
| Qwen 2.5 | 0.5B-72B | Great for multilingual |
| Mistral | 7B | High quality, well-tested |
| Gemma 2 | 2B, 9B | Efficient, Google's latest |
| Phi-3 | 3.8B | Microsoft, very efficient |

### Memory Requirements

With 4-bit quantization + LoRA:

| Model | VRAM (Training) | VRAM (Inference) |
|-------|----------------|------------------|
| 1B | ~4GB | ~2GB |
| 3B | ~8GB | ~4GB |
| 7B | ~16GB | ~8GB |
| 13B | ~24GB | ~12GB |

## Step 1: Install Unsloth

Unsloth has specific installation requirements. Run this cell to install:

In [ ]:
# Installation for CUDA 12.1+ (most modern systems)
# Uncomment and run if not already installed

# %pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# %pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kcgd567d/unsloth_abef80e4d4ea4a46a5da0161e1332b38
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kcgd567d/unsloth_abef80e4d4ea4a46a5da0161e1332b38
  Resolved https://github.com/unslothai/unsloth.git to commit 997f1a7ce5fb7a0319c2b8abe0e7af02e2160efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 124.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 15.8 MB/s eta 0:00:00

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
GPU Memory: 15.6 GB


In [3]:
# Import Unsloth
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

print(f"BFloat16 supported: {is_bfloat16_supported()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
BFloat16 supported: False


## Step 2: Choose Your Model

Unsloth provides pre-quantized 4-bit models for efficiency. Here are recommended options:

In [ ]:
# Model options - uncomment your choice

# Small models (4-8GB VRAM)
# model_name = "unsloth/Llama-3.2-1B-bnb-4bit"       # 1B params, ~4GB
# model_name = "unsloth/Qwen2.5-0.5B-bnb-4bit"      # 0.5B params, ~3GB

# Medium models (8-16GB VRAM)
model_name = "unsloth/Llama-3.2-3B-bnb-4bit"        # 3B params, ~8GB
# model_name = "unsloth/Qwen2.5-3B-bnb-4bit"        # 3B params, ~8GB
# model_name = "unsloth/gemma-2-2b-bnb-4bit"        # 2B params, ~6GB
# model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"  # 3.8B params, ~8GB

# Large models (16-24GB VRAM)
# model_name = "unsloth/Llama-3.1-8B-bnb-4bit"      # 8B params, ~16GB
# model_name = "unsloth/Mistral-7B-v0.3-bnb-4bit"   # 7B params, ~14GB
# model_name = "unsloth/Qwen2.5-7B-bnb-4bit"        # 7B params, ~14GB

# MOGON recommended very large Models (24GB VRAM or more)
# model_name = "/data/Ministral-3-14B-Instruct-2512"  # 14B params, ~24GB (requires 24GB VRAM or more)

print(f"Selected model: {model_name}")

Selected model: unsloth/Llama-3.2-3B-bnb-4bit


In [ ]:
# Instead we download/use the MOGON recommended model:
# Model configuration

import os
from pathlib import Path

local_model_path = Path("/data/Ministral-3-14B-Instruct-2512")
print("cwd:", Path.cwd())
print("os.name:", os.name)
print("resolved:", local_model_path.resolve())
print("exists:", local_model_path.exists())
print("is_dir:", local_model_path.is_dir())
if local_model_path.is_dir():
    print("Model found locally.")
    model_name = str(local_model_path)
else:
    from dotenv import load_dotenv
    from huggingface_hub import snapshot_download

    load_dotenv(Path(".env"))
    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        raise ValueError("HF_TOKEN not set. Add it to your .env file: HF_TOKEN=hf_your_token_here")

    print("Model not found locally. Downloading from Hugging Face...")
    model_name = snapshot_download(
        repo_id="mistralai/Ministral-3-14B-Instruct-2512",
        local_dir=str(local_model_path),
        token=hf_token,
    )

print(f"Selected model: {model_name}")

cwd: c:\Users\Morit\Desktop\Projekte\DeepLearning_internship_team3
os.name: nt
resolved: C:\Users\Morit\Desktop\Projekte\DeepLearning_internship_team3\data\Ministral-3-14B-Instruct-2512
exists: True
is_dir: True
Model found locally.
Selected model: data\Ministral-3-14B-Instruct-2512


## Exercise 11.1: Load the Model (20 min)

Load a 4-bit quantized model with Unsloth. The key parameters are:

- `model_name`: The HuggingFace model ID
- `max_seq_length`: Maximum context length (affects memory)
- `dtype`: Computation dtype (None = auto, or torch.float16/bfloat16)
- `load_in_4bit`: Whether to use 4-bit quantization

In [5]:
####solution 11.1
max_seq_length = 2048  # Can increase for longer contexts (uses more memory)
dtype = None  # Auto-detect (will use bfloat16 if supported)
load_in_4bit = True  # Use 4-bit quantization for memory efficiency

# TODO: Load the model using FastLanguageModel.from_pretrained()
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = ...,
#     max_seq_length = ...,
#     dtype = ...,
#     load_in_4bit = ...,
# )
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-bnb-4bit as a legacy tokenizer.


In [6]:
####test cell
# Verify model loaded correctly
print(f"Model type: {type(model).__name__}")
print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Max length: {max_seq_length}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params / 1e9:.2f}B")

# Check memory usage
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Model type: LlamaForCausalLM
Tokenizer type: TokenizersBackend
Vocab size: 128,000
Max length: 2048

Total parameters: 1.80B
GPU memory allocated: 2.29 GB
GPU memory reserved: 2.32 GB


## Exercise 11.2: Add LoRA Adapters (20 min)

Now add LoRA adapters to the model. Key parameters:

- `r`: LoRA rank (8-64, higher = more capacity)
- `lora_alpha`: Scaling factor (typically 16-32)
- `lora_dropout`: Dropout for regularization (0-0.1)
- `target_modules`: Which layers to apply LoRA to

In [7]:
####solution 11.2
# LoRA configuration
lora_config = {
    "r": 16,  # Rank - start with 16, increase if underfitting
    "lora_alpha": 32,  # Scaling factor, commonly 2*r
    "lora_dropout": 0.05,  # Small dropout for regularization
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",  # MLP (optional)
    ],
    "bias": "none",  # Don't train biases
    "use_gradient_checkpointing": "unsloth",  # Memory optimization
    "random_state": 42,
}

# TODO: Add LoRA adapters using FastLanguageModel.get_peft_model()
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = lora_config["r"],
#     target_modules = lora_config["target_modules"],
#     lora_alpha = lora_config["lora_alpha"],
#     lora_dropout = lora_config["lora_dropout"],
#     bias = lora_config["bias"],
#     use_gradient_checkpointing = lora_config["use_gradient_checkpointing"],
#     random_state = lora_config["random_state"],
# )
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_config["r"],
    target_modules=lora_config["target_modules"],
    lora_alpha=lora_config["lora_alpha"],
    lora_dropout=lora_config["lora_dropout"],
    bias=lora_config["bias"],
    use_gradient_checkpointing=lora_config["use_gradient_checkpointing"],
    random_state=lora_config["random_state"],
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [8]:
####test cell
# Verify LoRA was added
print("Model structure with LoRA:")
model.print_trainable_parameters()

# Detailed parameter breakdown
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

Model structure with LoRA:
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511

Trainable: 24,313,856 / 1,827,777,536 (1.3302%)


## Exercise 11.3: Test the Model (15 min)

Before training, let's test that the model works correctly.

In [ ]:
####solution 11.3

test_prompt = "The capital of France is"
# TODO: Create a simple test prompt and generate a response
#
# 1. Put model in inference mode: FastLanguageModel.for_inference(model)
FastLanguageModel.for_inference(model)
# 2. Tokenize a prompt: inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
# 3. Generate: outputs = model.generate(**inputs, max_new_tokens=64)
outputs = model.generate(**inputs, 
                         max_new_tokens=64,
                         temperature=0.7,  # Optional: adjust for more creative (higher) or focused (lower) responses
                         do_sample=True,  # Enable sampling for more varied responses
)
# 4. Decode: response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Prompt: {test_prompt}")
print(f"Response: {response}")

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

Prompt: The capital of France is
Response: The capital of France is Paris, the country’s largest city, which is home to the Louvre Museum and the Eiffel Tower. The city is also home to the Palace of Versailles, a former royal residence. The city is also home to the Château de Versailles, a former royal residence.
The country’s second largest


In [11]:
####test cell
# Test with a chat-style prompt (for instruct models)
# Different models use different chat templates

# Llama 3 style
llama3_prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

What is machine learning?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# Qwen style
qwen_prompt = """<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
"""

# Mistral/Generic style
generic_prompt = """[INST] What is machine learning? [/INST]"""

# Choose based on your model
if "llama" in model_name.lower() or "Llama" in model_name:
    chat_prompt = llama3_prompt
elif "qwen" in model_name.lower() or "Qwen" in model_name:
    chat_prompt = qwen_prompt
else:
    chat_prompt = generic_prompt

print(f"Using prompt format for: {model_name}")
print(f"Prompt:\n{chat_prompt}")

Using prompt format for: unsloth/Llama-3.2-3B-bnb-4bit
Prompt:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

What is machine learning?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [12]:
####test cell
#Generate response with chat prompt
FastLanguageModel.for_inference(model)

inputs = tokenizer(chat_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Response:")
print(response)

Response:
user

What is machine learning?assistant

What is machine learning?　　　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　 　

## Understanding Model Architecture

Let's explore the model structure to understand where LoRA is applied.

In [13]:
# Print model architecture (first few layers)
def print_model_structure(model, max_depth=3):
    """Print model structure with LoRA layers highlighted."""
    for name, module in model.named_modules():
        depth = name.count('.')
        if depth <= max_depth:
            indent = '  ' * depth
            module_type = type(module).__name__

            # Highlight LoRA layers
            if 'lora' in module_type.lower():
                print(f"{indent}{name}: {module_type} [LoRA]")
            elif depth <= 2:
                print(f"{indent}{name}: {module_type}")

print("Model structure (showing LoRA layers):")
print_model_structure(model)

Model structure (showing LoRA layers):
: PeftModelForCausalLM
base_model: LoraModel [LoRA]
  base_model.model: LlamaForCausalLM
    base_model.model.model: LlamaModel
    base_model.model.lm_head: Linear


In [14]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

In [15]:
# Check which modules have LoRA adapters
lora_modules = []
for name, module in model.named_modules():
    if hasattr(module, 'lora_A'):
        lora_modules.append(name)

print(f"Found {len(lora_modules)} LoRA modules:")
for name in lora_modules[:10]:  # Show first 10
    print(f"  - {name}")
if len(lora_modules) > 10:
    print(f"  ... and {len(lora_modules) - 10} more")

Found 196 LoRA modules:
  - base_model.model.model.layers.0.self_attn.q_proj
  - base_model.model.model.layers.0.self_attn.k_proj
  - base_model.model.model.layers.0.self_attn.v_proj
  - base_model.model.model.layers.0.self_attn.o_proj
  - base_model.model.model.layers.0.mlp.gate_proj
  - base_model.model.model.layers.0.mlp.up_proj
  - base_model.model.model.layers.0.mlp.down_proj
  - base_model.model.model.layers.1.self_attn.q_proj
  - base_model.model.model.layers.1.self_attn.k_proj
  - base_model.model.model.layers.1.self_attn.v_proj
  ... and 186 more


## Memory Optimization Tips

If you're running low on GPU memory:

1. **Reduce max_seq_length**: 1024 instead of 2048
2. **Use a smaller model**: Llama-3.2-1B instead of 3B
3. **Reduce LoRA rank**: r=8 instead of r=16
4. **Target fewer modules**: Just q_proj and v_proj
5. **Enable gradient checkpointing**: Already done with `use_gradient_checkpointing="unsloth"`

In [16]:
# Memory usage summary
if torch.cuda.is_available():
    print("GPU Memory Summary:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"  Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.2f} GB")
    print(f"  Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU Memory Summary:
  Allocated: 2.53 GB
  Reserved: 2.76 GB
  Free: 12.87 GB
  Total: 15.64 GB


## Summary

### What You Learned

1. **Unsloth Benefits**: 2x faster, 70% less memory
2. **Model Loading**: Use `FastLanguageModel.from_pretrained()` with 4-bit quantization
3. **Adding LoRA**: Use `FastLanguageModel.get_peft_model()` with target modules
4. **Chat Templates**: Different models use different prompt formats

### Key Parameters

| Parameter | Recommended Value | Notes |
|-----------|------------------|-------|
| `r` (rank) | 8-32 | Higher = more capacity |
| `lora_alpha` | 2 * r | Scaling factor |
| `target_modules` | q,k,v,o projections | More = better but slower |
| `max_seq_length` | 1024-4096 | Longer = more memory |

### Next: Data Preparation

In Notebook 12, we'll prepare our training data with proper chat formatting!